In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# INFERENCE

In [ ]:
!pip install imagehash # Install missing library

In [ ]:
import os
import time
import numpy as np
import torch
from torchvision import models, transforms
from torchvision.ops import nms
from PIL import Image, ImageDraw, ImageFont
import matplotlib.cm as cm  # Fixed: Correct import for colormap

import imagehash
from skimage.metrics import structural_similarity as ssim

print("--- Step 1: Differential detection pipeline configuration ---")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ===================================================================
# 1️⃣ LOAD MODEL
# ===================================================================
model_path = "/content/drive/MyDrive/Colab Notebooks/best_resnet50_pcb_defects_50epochs.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found: {model_path}")

checkpoint = torch.load(model_path, map_location=device)

class_names = checkpoint.get("class_names", None)
if class_names is None:
    class_names = ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']

num_classes = len(class_names)
print(f"Classes: {class_names}")

defect_classifier = models.resnet50(weights=None)
in_features = defect_classifier.fc.in_features
defect_classifier.fc = torch.nn.Linear(in_features, num_classes)

defect_classifier.load_state_dict(checkpoint["model_state_dict"])
defect_classifier.to(device)
defect_classifier.eval()

# ===================================================================
# 2️⃣ CONFIG (Made configurable)
# ===================================================================
golden_images_dir = "/content/drive/MyDrive/PCB_DATASET/PCB_USED/"
input_folder = "/content/drive/MyDrive/PCB_DATASET/images/Mouse_bite"
output_dir = "/content/drive/MyDrive/PCB_DATASET/Inference_Results/Check"

# Check directories
if not os.path.exists(golden_images_dir):
    raise FileNotFoundError(f"Golden PCB folder missing: {golden_images_dir}")
if not os.path.exists(input_folder):
    raise FileNotFoundError(f"Input folder missing: {input_folder}")
os.makedirs(output_dir, exist_ok=True)

# Parameters (adjust as needed)
WINDOW_SIZE = 128
STRIDE = WINDOW_SIZE // 4
SIMILARITY_THRESHOLD = 0.95
CLASSIFIER_CONFIDENCE_THRESHOLD = 0.80
MIN_IMAGE_SIZE = WINDOW_SIZE  # Minimum size to process

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ===================================================================
# 3️⃣ LOAD GOLDEN DB
# ===================================================================
def create_golden_image_database(golden_dir: str) -> list:
    db = []
    print("Loading Golden Images...")
    for fname in os.listdir(golden_dir):
        path = os.path.join(golden_dir, fname)
        if os.path.isfile(path) and fname.lower().endswith(('.jpg', '.png', '.jpeg')):
            try:
                img = Image.open(path).convert("RGB")
                h = imagehash.phash(img)
                db.append({"filename": fname, "image": img, "hash": h})
            except Exception as e:
                print(f"Warning: Failed to load golden image {fname}: {e}")
    print(f"Golden images loaded: {len(db)}")
    return db

golden_db = create_golden_image_database(golden_images_dir)
if not golden_db:
    raise ValueError("No golden images loaded. Check the directory.")

# ===================================================================
# 4️⃣ FIND BEST GOLDEN MATCH
# ===================================================================
def find_best_match(input_image: Image.Image, golden_database: list) -> Image.Image:
    if not golden_database:
        raise ValueError("Golden database is empty.")

    ihash = imagehash.phash(input_image)
    best = min(golden_database, key=lambda x: ihash - x["hash"])
    print(f"Golden Matched: {best['filename']}")

    # Resize golden image to match input size for consistency
    golden_resized = best["image"].resize(input_image.size)
    return golden_resized

# ===================================================================
# 5️⃣ DETECT ANOMALIES
# ===================================================================
def detect_anomalies_by_comparison(input_image: Image.Image, golden_image: Image.Image) -> list:
    # Ensure sizes match (should be handled in matching, but double-check)
    if input_image.size != golden_image.size:
        input_image = input_image.resize(golden_image.size)

    detections = []
    w, h = input_image.size

    # Skip if image is too small
    if w < MIN_IMAGE_SIZE or h < MIN_IMAGE_SIZE:
        print("Image too small for windowing. Skipping.")
        return []

    print("Processing sliding windows...")
    start_time = time.time()

    for y in range(0, h - WINDOW_SIZE + 1, STRIDE):
        for x in range(0, w - WINDOW_SIZE + 1, STRIDE):
            win_in = input_image.crop((x, y, x + WINDOW_SIZE, y + WINDOW_SIZE))
            win_gold = golden_image.crop((x, y, x + WINDOW_SIZE, y + WINDOW_SIZE))

            # Convert to grayscale for SSIM
            g1 = np.array(win_in.convert("L"))
            g2 = np.array(win_gold.convert("L"))

            score, _ = ssim(g2, g1, full=True)

            if score < SIMILARITY_THRESHOLD:
                patch = inference_transform(win_in).unsqueeze(0).to(device)

                with torch.no_grad():
                    out = defect_classifier(patch)
                    prob = torch.softmax(out, dim=1)
                    conf, idx = torch.max(prob, 1)

                if conf.item() >= CLASSIFIER_CONFIDENCE_THRESHOLD:
                    detections.append({
                        "box": [x, y, x + WINDOW_SIZE, y + WINDOW_SIZE],
                        "label": class_names[idx.item()],
                        "confidence": conf.item()
                    })

    print(f"Raw detections: {len(detections)} (Time: {time.time() - start_time:.2f}s)")

    if not detections:
        return []

    # Apply NMS
    boxes = torch.tensor([d["box"] for d in detections], dtype=torch.float32)
    scores = torch.tensor([d["confidence"] for d in detections], dtype=torch.float32)

    keep = nms(boxes, scores, 0.2)
    final = [detections[i] for i in keep]

    print(f"Final detections after NMS: {len(final)}")
    return final

# ===================================================================
# 6️⃣ DRAW RESULT
# ===================================================================
def draw_detections_on_image(image: Image.Image, detections: list) -> Image.Image:
    img = image.copy()
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 32)
    except:
        font = ImageFont.load_default()

    # Use a single color (red) for all defect boxes
    color = (255, 0, 0)  # Red

    for det in detections:
        x1, y1, x2, y2 = det["box"]
        label = det["label"]
        conf = det["confidence"]

        draw.rectangle((x1, y1, x2, y2), outline=color, width=5)

        text = f"{label} ({conf:.2f})"
        draw.text((x1, y1 - 25), text, fill="white", font=font)

    return img

# ===================================================================
# 7️⃣ RUN ON FULL FOLDER
# ===================================================================
images = [f for f in os.listdir(input_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

print(f"\nFound {len(images)} images to process.\n")

processed = 0
errors = 0

for img_name in images:
    try:
        print(f"\nProcessing: {img_name}")
        path = os.path.join(input_folder, img_name)
        input_img = Image.open(path).convert("RGB")

        golden = find_best_match(input_img, golden_db)

        detections = detect_anomalies_by_comparison(input_img, golden)

        result = draw_detections_on_image(input_img, detections)

        # Save image only (no txt file)
        save_path = os.path.join(output_dir, f"result_{img_name}")
        result.save(save_path)

        print(f"Saved: {save_path}")
        processed += 1

    except Exception as e:
        print(f"Error processing {img_name}: {e}")
        errors += 1

print(f"\nDONE ✅ Processed: {processed}, Errors: {errors}")